In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4" 

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader
from vllm import LLM, SamplingParams

In [9]:
data_idx = 150

In [3]:
model_path = "/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/pos_rs_0.1-neg_rs_0.01-equ+belu+rule-piecewise_error/actor/global_step_300"

tokenizer = AutoTokenizer.from_pretrained(model_path)
torch.cuda.empty_cache()
base_model = LLM(
    model=model_path,
    tensor_parallel_size=1,  # 使用全部8张GPU
    gpu_memory_utilization=0.85,  # 可以设置更高的内存利用率
    dtype="auto"
)

INFO 08-17 15:08:42 config.py:1450] Downcasting torch.float32 to torch.float16.
INFO 08-17 15:08:42 llm_engine.py:174] Initializing an LLM engine (v0.5.4) with config: model='/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/pos_rs_0.1-neg_rs_0.01-equ+belu+rule-piecewise_error/actor/global_step_300', speculative_config=None, tokenizer='/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/pos_rs_0.1-neg_rs_0.01-equ+belu+rule-piecewise_error/actor/global_step_300', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityCo

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]


INFO 08-17 15:08:53 model_runner.py:732] Loading model weights took 14.2448 GB
INFO 08-17 15:08:54 gpu_executor.py:102] # GPU blocks: 59353, # CPU blocks: 4681
INFO 08-17 15:08:57 model_runner.py:1024] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 08-17 15:08:57 model_runner.py:1028] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 08-17 15:09:09 model_runner.py:1225] Graph capturing finished in 12 secs.


In [15]:
train_data_path = "dataset/openr1.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=5,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)
test_data_path = "dataset/valid.all.parquet"
test_dataset = RLHFDataset(parquet_files=test_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

test_dataloader = DataLoader(dataset=test_dataset,
                            batch_size=1,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 45792
filter dataset len: 45764
original dataset len: 6023
filter dataset len: 6021


In [17]:
test_data = test_dataset[data_idx]
# print(test_data['reward_model'])
input_text = tokenizer.decode(test_data['input_ids'], skip_special_tokens=True)

In [18]:
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=1.0,
    max_tokens=8192,
)

prompts = [input_text] * 8
outputs = base_model.generate(prompts, sampling_params)

Processed prompts: 100%|██████████| 8/8 [00:57<00:00,  7.14s/it, est. speed input: 53.12 toks/s, output: 379.32 toks/s]


In [13]:
group_rollout = []
for output in outputs:
    # 提取生成的文本
    full_text = output.outputs[0].text
    # 只保留输入之后新生成的部分
    generated_text = full_text[len(input_text):]
    group_rollout.append(generated_text)

In [14]:
print(test_data['reward_model'])
for i in range(len(group_rollout)):
    print(f"********{i}**********")
    print(group_rollout[i])  # 从列表中提取文本
    print("__________end_____________\n")

{'ground_truth': '1', 'style': 'rule'}
********0**********
) = f(0,4) = 0$ (from the base case, since $f(1,2) = 4$).
   - $f(1,4) = f(0, f(1,3)) = f(0,0) = 1$ (from the base case, since $f(1,3) = 0$).

So, for $i = 1$, we have:
   \[
   \begin{aligned}
   f(1,0) &= 2, \\
   f(1,1) &= 3, \\
   f(1,2) &= 4, \\
   f(1,3) &= 0, \\
   f(1,4) &= 1.
   \end{aligned}
   \]

**Case $i = 2$:**
   - $f(2,0) = f(1,1) = 3$.
   - $f(2,1) = f(1, f(2,0)) = f(1,3) = 0$.
   - $f(2,2) = f(1, f(2,1)) = f(1,0) = 2$.
   - $f(2,3) = f(1, f(2,2)) = f(1,2) = 4$.
   - $f(2,4) = f(1, f(2,3)) = f(1,4) = 1$.

So, for $i = 2$, we have:
   \[
   \begin{aligned}
   f(2,0) &= 3, \\
   f(2,1) &= 0, \\
   f(2,2) &= 2, \\
   f(2,3) &= 4, \\
   f(2,4) &= 1.
   \end{aligned}
   \]

**Case $i = 3$:**
   - $f(3,0) = f(2,1) = 0$.
   - $f(3,1) = f(2, f(3,0)) = f(2,0) = 3$.
   - $f(3,2) = f(2, f(3,1)) = f(2,3) = 4$.
   - $f(3,3) = f(2, f(3,2)) = f(2,4) = 1$.
   - $f(3,4) = f(2, f(3,3)) = f(2,1) = 0$.

So, for $i = 3$, we have:
